# WaveForge — Brain Haemorrhage Dataset Generator

**Run All — fully automated. Check Cell 4 plots BEFORE proceeding to generation.**

### Workflow
- **Cell 3:** Generate 4 validation samples (1 per class, ~5 min)
- **Cell 4:** 🔍 **VISUAL CHECK** — material maps, Ez fields, signals, DAS images
  - Confirm tissue layers visible, signals non-zero, DAS shows hotspot near bleed
  - If anything looks wrong — **stop here and debug before running full generation**
- **Cell 5:** Both GPUs generate training samples (800 each → 1600 total)
- **Cell 6:** Both GPUs generate test samples (200 each → 400 total)

| Property | Value |
|----------|-------|
| Frequency | 1.0 GHz | Grid | 64³ at 3mm/cell |
| Antennas | 8-element ring | **Steps** | **700** (validated: full round-trip) |
| Classes | 0=healthy 1=epidural 2=subdural 3=intracerebral |

**Accelerator:** GPU T4 x2 | **Est. total:** ~13h (700 steps × 2 phases)

In [ ]:
# ── CELL 1: Setup ─────────────────────────────────────────────────────────
import subprocess, sys, os, pathlib, threading, time, json, datetime
import math

REPO_URL = 'https://github.com/shahzaibshazoo/waveforge.git'
REPO_DIR = pathlib.Path('/kaggle/working/waveforge')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

src_path = str(REPO_DIR / 'src')
if src_path not in sys.path: sys.path.insert(0, src_path)
os.chdir(REPO_DIR)

import torch, numpy as np
assert torch.cuda.is_available(), 'No GPU — enable T4 x2!'
N_GPUS = torch.cuda.device_count()
print(f'GPUs: {N_GPUS}')
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f'  [{i}] {p.name}  {p.total_memory/1e9:.1f} GB')
print(f'PyTorch: {torch.__version__}\n✅ Ready')

In [ ]:
# ── CELL 2: Configuration ─────────────────────────────────────────────────
FREQ_GHZ    = 1.0
GRID_SIZE   = 64
DX_MM       = 3.0
N_TX        = 8
RING_RADIUS = 30
N_STEPS     = 700     # VALIDATED: 3.96ns covers full 2.94ns round-trip in head

N_TRAIN_TOTAL   = 1600;  N_TRAIN_PER_GPU = N_TRAIN_TOTAL // 2
N_TEST_TOTAL    = 400;   N_TEST_PER_GPU  = N_TEST_TOTAL  // 2

OUTPUT_ROOT  = pathlib.Path('/kaggle/working/brain_haemorrhage_dataset')
TRAIN_DIR_G0 = OUTPUT_ROOT / 'train_gpu0'
TRAIN_DIR_G1 = OUTPUT_ROOT / 'train_gpu1'
TEST_DIR_G0  = OUTPUT_ROOT / 'test_gpu0'
TEST_DIR_G1  = OUTPUT_ROOT / 'test_gpu1'
TRAIN_DIR    = OUTPUT_ROOT / 'train'
TEST_DIR     = OUTPUT_ROOT / 'test'

GPU0 = 'cuda:0';  GPU1 = 'cuda:1' if N_GPUS > 1 else 'cuda:0'

TRAIN_SEED_G0 = 0;         TRAIN_SEED_G1 = 100_000
TEST_SEED_G0  = 10_000_000; TEST_SEED_G1  = 10_100_000

sec_per = N_TX * N_STEPS * GRID_SIZE**3 / 62e6 * 2
print(f'~{sec_per:.0f}s/sample  →  train: ~{sec_per*N_TRAIN_PER_GPU/3600:.1f}h + test: ~{sec_per*N_TEST_PER_GPU/3600:.1f}h')

In [ ]:
# ── CELL 3: Generate 4 validation samples (one per class, ~5 min) ─────────
from datasets.generator import BrainDatasetGenerator

print('Generating 4 validation samples (one per class)...')
val_gen = BrainDatasetGenerator(
    output_dir=str(OUTPUT_ROOT / 'validation'),
    freq_hz=FREQ_GHZ*1e9, grid_size=GRID_SIZE, dx_mm=DX_MM,
    n_tx=N_TX, ring_radius_cells=RING_RADIUS, n_steps=N_STEPS,
    device=GPU0,
)
val_manifest = val_gen.generate_balanced_dataset(
    n_samples=4, phantom_id='train', base_seed=42, show_progress=True
)
if val_manifest['n_completed'] < 2:
    raise RuntimeError('Validation failed — fewer than 2 samples generated.')
print(f'\n{val_manifest["n_completed"]}/4 samples generated')
print(f'Classes: {val_manifest["class_counts"]}')
print('✅ Proceed to Cell 4 to inspect the samples visually')

In [ ]:
# ── CELL 4: 🔍 VISUAL CHECK — inspect before full generation ──────────────
#
# Shows for each of the 4 validation samples:
#   Col 1: Material map (eps_r) — tissue layers + bleed location
#   Col 2: Scattered signals from TX[0] — should be non-zero for bleed classes
#   Col 3: DAS backprojection — hotspot should be near true bleed centre
#
# IF any of these look wrong → STOP HERE. Do not proceed to Cell 5.

import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt

label_to_name = {0:'Healthy', 1:'Epidural', 2:'Subdural', 3:'Intracerebral'}

# Sort samples by label so rows are always in order 0→3
sorted_samples = sorted(
    [(lbl, path) for lbl, path in zip(val_manifest['labels'], val_manifest['sample_paths'])],
    key=lambda x: x[0]
)

fig, axes = plt.subplots(len(sorted_samples), 3, figsize=(18, 5 * len(sorted_samples)))
if len(sorted_samples) == 1: axes = axes[np.newaxis, :]
fig.suptitle(
    f'🔍 VALIDATION CHECK — {len(sorted_samples)} samples, {N_TX} antennas, {N_STEPS} steps\n'
    f'Confirm: layers visible | signals non-zero | DAS near true bleed',
    fontsize=13, fontweight='bold', color='darkblue'
)

ext_mm = [0, GRID_SIZE*DX_MM, 0, GRID_SIZE*DX_MM]

for row, (label, path) in enumerate(sorted_samples):
    s     = np.load(path, allow_pickle=True)
    name  = label_to_name.get(int(label), str(label))
    age   = str(s['bleed_age'])
    r_mm  = float(s['bleed_radius_mm'])
    scat  = s['signals_scattered']    # (N_tx, N_rx, N_steps)
    t_ns  = np.arange(scat.shape[2]) * float(s['dt_s']) * 1e9
    skull_r = int(s['phantom_skull_inner_r'])
    gray_r  = int(s['phantom_gray_r'])

    # ── Panel 1: synthetic material map ──────────────────────────────────
    ax = axes[row, 0]
    eps = np.ones((GRID_SIZE, GRID_SIZE), dtype=np.float32)
    I, J = np.meshgrid(np.arange(GRID_SIZE), np.arange(GRID_SIZE))
    cx = cy = GRID_SIZE // 2

    def smask(r): return (I - cx)**2 + (J - cy)**2 <= r**2

    scalp_r = skull_r + 2
    eps[smask(scalp_r)] = 40.0
    eps[smask(skull_r)] = 13.0
    eps[smask(gray_r)]  = 52.7
    eps[smask(max(gray_r - 7, 3))] = 38.1   # white matter core

    if r_mm > 0:
        bc = [int(s['bleed_center_cells'][i]) for i in range(2)]
        br = int(s['bleed_radius_cells'])
        eps[(I - bc[0])**2 + (J - bc[1])**2 <= br**2] = 61.0  # bleed

    im = ax.imshow(eps.T, origin='lower', cmap='jet', vmin=1, vmax=70,
                   extent=ext_mm, aspect='auto')
    # Mark antenna ring
    for k in range(N_TX):
        ang = 2 * math.pi * k / N_TX
        ax.plot((cx + RING_RADIUS*math.cos(ang))*DX_MM,
                (cy + RING_RADIUS*math.sin(ang))*DX_MM,
                'w^', markersize=6)
    if r_mm > 0:
        ax.plot(float(s['bleed_center_mm'][0]), float(s['bleed_center_mm'][1]),
                'r*', markersize=14, markeredgecolor='white')
    plt.colorbar(im, ax=ax, label='ε_r')
    ax.set(title=f'{name} — material map (skull_r={skull_r}, gray_r={gray_r})',
           xlabel='x (mm)', ylabel='y (mm)')

    # ── Panel 2: scattered signals ────────────────────────────────────────
    ax2 = axes[row, 1]
    energy = float((scat**2).sum())
    colors = plt.cm.tab10(np.linspace(0, 1, min(scat.shape[1], 8)))
    for rx in range(min(scat.shape[1], 8)):
        ax2.plot(t_ns, scat[0, rx], color=colors[rx % len(colors)], alpha=0.7, lw=1)
    ax2.set(title=f'Scattered signals TX[0]  ΔE={energy:.2e}',
            xlabel='Time (ns)', ylabel='Ez (V/m)')
    ax2.grid(alpha=0.3)
    # Flag if energy is suspiciously low for a bleed
    if r_mm > 0 and energy < 1e-10:
        ax2.set_facecolor('#fff0f0')
        ax2.set_title(ax2.get_title() + ' ⚠ LOW', color='red')

    # ── Panel 3: DAS backprojection ───────────────────────────────────────
    ax3 = axes[row, 2]
    das = s['das_image']
    im3 = ax3.imshow(das.T, origin='lower', cmap='hot', extent=ext_mm, aspect='auto')
    plt.colorbar(im3, ax=ax3, label='DAS power')
    if r_mm > 0:
        bx = float(s['bleed_center_mm'][0]); by = float(s['bleed_center_mm'][1])
        ax3.plot(bx, by, 'c+', markersize=18, mew=2.5, label=f'true ({bx:.0f},{by:.0f})mm')
        # Find DAS peak
        iy, ix = np.unravel_index(np.argmax(das), das.shape)
        px = ix / das.shape[0] * GRID_SIZE * DX_MM
        py = iy / das.shape[1] * GRID_SIZE * DX_MM
        err = math.sqrt((px-bx)**2 + (py-by)**2)
        ax3.plot(px, py, 'y^', markersize=12, mew=1.5, label=f'DAS peak (err={err:.0f}mm)')
        ax3.legend(fontsize=7)
    ax3.set(title=f'DAS backprojection — {name}', xlabel='x (mm)', ylabel='y (mm)')
    # Green/red border
    detected = float((scat**2).sum()) > 1e-9
    ok = (detected == (r_mm > 0))
    for sp in ax3.spines.values():
        sp.set_edgecolor('#2ecc71' if ok else '#e74c3c'); sp.set_linewidth(3)

plt.tight_layout()
os.makedirs('docs/assets', exist_ok=True)
plt.savefig('docs/assets/validation_check.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → docs/assets/validation_check.png')
print()
print('✅ If all 4 rows look correct → proceed to Cell 5 for full generation')
print('❌ If any row looks wrong → debug before running 10+ hours of generation')

In [ ]:
# ── CELL 5: Phase 1 — both GPUs generate TRAINING samples ─────────────────
results = {}; errors = {}

def run_train_gpu(gpu, out_dir, seed, n):
    try:
        print(f'[{gpu}] Starting {n} train samples...')
        gen = BrainDatasetGenerator(
            output_dir=str(out_dir),
            freq_hz=FREQ_GHZ*1e9, grid_size=GRID_SIZE, dx_mm=DX_MM,
            n_tx=N_TX, ring_radius_cells=RING_RADIUS, n_steps=N_STEPS,
            device=gpu, seed=seed,
        )
        results[gpu] = gen.generate_balanced_dataset(
            n_samples=n, phantom_id='train', base_seed=seed, show_progress=True)
        print(f'[{gpu}] Done: {results[gpu]["n_completed"]} samples')
    except Exception as e:
        errors[gpu] = e; print(f'[{gpu}] ERROR: {e}')

t0 = time.time()
threads = [
    threading.Thread(target=run_train_gpu, args=(GPU0, TRAIN_DIR_G0, TRAIN_SEED_G0, N_TRAIN_PER_GPU)),
    threading.Thread(target=run_train_gpu, args=(GPU1, TRAIN_DIR_G1, TRAIN_SEED_G1, N_TRAIN_PER_GPU)),
]
for t in threads: t.start()
for t in threads: t.join()
if errors: raise RuntimeError(f'Errors: {errors}')

import shutil
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
train_paths, train_labels, train_types, train_ages = [], [], [], []
train_class_counts = {0:0, 1:0, 2:0, 3:0}
for gk, sd in [('cuda:0', TRAIN_DIR_G0), ('cuda:1', TRAIN_DIR_G1)]:
    m = results[gk]
    for op, lb, bt, ba in zip(m['sample_paths'], m['labels'], m['bleed_types'], m['bleed_ages']):
        ni = len(train_paths)
        np_ = TRAIN_DIR / f'sample_{ni:06d}.npz'
        shutil.copy(op, np_)
        train_paths.append(str(np_)); train_labels.append(lb)
        train_types.append(bt); train_ages.append(ba)
        train_class_counts[lb] += 1
train_manifest = {
    'n_completed': len(train_paths), 'n_failed': 0,
    'class_counts': train_class_counts, 'sample_paths': train_paths,
    'labels': train_labels, 'bleed_types': train_types, 'bleed_ages': train_ages,
}
print(f'\n✅ Phase 1 done in {(time.time()-t0)/3600:.2f}h  |  {len(train_paths)} samples')
print(f'   Classes: {train_class_counts}')

In [ ]:
# ── CELL 6: Phase 2 — both GPUs generate TEST samples ─────────────────────
test_results = {}; test_errors = {}

def run_test_gpu(gpu, out_dir, seed, n):
    try:
        print(f'[{gpu}] Starting {n} test samples (seed space 10M+)...')
        gen = BrainDatasetGenerator(
            output_dir=str(out_dir),
            freq_hz=FREQ_GHZ*1e9, grid_size=GRID_SIZE, dx_mm=DX_MM,
            n_tx=N_TX, ring_radius_cells=RING_RADIUS, n_steps=N_STEPS,
            device=gpu, seed=seed,
        )
        test_results[gpu] = gen.generate_balanced_dataset(
            n_samples=n, phantom_id='test', base_seed=seed, show_progress=True)
        print(f'[{gpu}] Done: {test_results[gpu]["n_completed"]} samples')
    except Exception as e:
        test_errors[gpu] = e; print(f'[{gpu}] ERROR: {e}')

t0 = time.time()
threads = [
    threading.Thread(target=run_test_gpu, args=(GPU0, TEST_DIR_G0, TEST_SEED_G0, N_TEST_PER_GPU)),
    threading.Thread(target=run_test_gpu, args=(GPU1, TEST_DIR_G1, TEST_SEED_G1, N_TEST_PER_GPU)),
]
for t in threads: t.start()
for t in threads: t.join()
if test_errors: raise RuntimeError(f'Errors: {test_errors}')

TEST_DIR.mkdir(parents=True, exist_ok=True)
test_paths, test_labels, test_types, test_ages = [], [], [], []
test_class_counts = {0:0, 1:0, 2:0, 3:0}
for gk, sd in [('cuda:0', TEST_DIR_G0), ('cuda:1', TEST_DIR_G1)]:
    m = test_results[gk]
    for op, lb, bt, ba in zip(m['sample_paths'], m['labels'], m['bleed_types'], m['bleed_ages']):
        ni = len(test_paths)
        np_ = TEST_DIR / f'sample_{ni:06d}.npz'
        shutil.copy(op, np_)
        test_paths.append(str(np_)); test_labels.append(lb)
        test_types.append(bt); test_ages.append(ba)
        test_class_counts[lb] += 1
test_manifest = {
    'n_completed': len(test_paths), 'class_counts': test_class_counts,
    'sample_paths': test_paths, 'labels': test_labels,
    'bleed_types': test_types, 'bleed_ages': test_ages,
}
print(f'\n✅ Phase 2 done in {(time.time()-t0)/3600:.2f}h  |  {len(test_paths)} test samples')
print(f'   Classes: {test_class_counts}')

In [ ]:
# ── CELL 7: Save master manifest ──────────────────────────────────────────
try: commit = subprocess.check_output(['git','-C',str(REPO_DIR),'rev-parse','--short','HEAD'],text=True).strip()
except: commit = 'unknown'

master = {
    'version': '1.3', 'created_at': datetime.datetime.now().isoformat(),
    'waveforge_commit': commit,
    'n_total': len(train_paths)+len(test_paths),
    'n_train': len(train_paths), 'n_test': len(test_paths),
    'train_class_counts': train_class_counts,
    'test_class_counts':  test_class_counts,
    'class_names': {0:'healthy',1:'epidural',2:'subdural',3:'intracerebral'},
    'phantom_design': 'unique_per_sample',
    'freq_hz': FREQ_GHZ*1e9, 'grid_shape': [GRID_SIZE]*3,
    'dx_mm': DX_MM, 'n_tx': N_TX, 'n_steps': N_STEPS,
    'notes': f'700 steps validated: round-trip covers 520 steps needed for deep ICH',
    'train_dir': str(TRAIN_DIR), 'test_dir': str(TEST_DIR),
}
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
manifest_path = OUTPUT_ROOT / 'dataset_manifest.json'
with open(manifest_path,'w') as f: json.dump(master,f,indent=2)
print(f'Total: {master["n_total"]}  |  commit: {commit}')

In [ ]:
# ── CELL 8: Package for download ──────────────────────────────────────────
import shutil
OUT = pathlib.Path('/kaggle/working/waveforge_brain_outputs')
OUT.mkdir(exist_ok=True)
shutil.copy(manifest_path, OUT)
for img in ['docs/assets/validation_check.png', 'docs/assets/brain_dataset_samples.png']:
    if pathlib.Path(img).exists(): shutil.copy(img, OUT)
train_f = sorted(TRAIN_DIR.glob('*.npz'))
test_f  = sorted(TEST_DIR.glob('*.npz'))
mb = sum(f.stat().st_size for f in train_f+test_f)/1e6
print(f'Train: {len(train_f)} | Test: {len(test_f)} | Size: {mb:.1f} MB')
print('🏁 Done!')